# Clone Repository and set up the Environment

In [1]:
!pwd

/content


In [2]:
!git clone https://github.com/VictoryChianumba/robust-eeg-models

fatal: destination path 'robust-eeg-models' already exists and is not an empty directory.


In [3]:
%cd robust-eeg-models

/content/robust-eeg-models


In [50]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	cache/

nothing added to commit but untracked files present (use "git add" to track)


In [6]:
!git config --global user.email "chianumbav@gmial.com"
!git config --global user.name "Victory Chianumba"

In [7]:
# 1️⃣ Upgrade the package manager
!pip install --upgrade --quiet pip

!pip install torch torchvision torchaudio
!pip install mne moabb
!pip install torch-lr-finder
!pip install optuna

# Clean uninstall
!pip uninstall -y braindecode

# Install latest code from GitHub (which includes CTNet)
!pip install git+https://github.com/braindecode/braindecode.git@master --no-cache-dir

Found existing installation: braindecode 1.2.0
Uninstalling braindecode-1.2.0:
  Successfully uninstalled braindecode-1.2.0
  Cloning https://github.com/braindecode/braindecode.git (to revision master) to /tmp/pip-req-build-i3l3igdg
  Running command git clone --filter=blob:none --quiet https://github.com/braindecode/braindecode.git /tmp/pip-req-build-i3l3igdg
  Resolved https://github.com/braindecode/braindecode.git to commit 13b0a2c174b42cc065a53c28f8e225b84e37a84e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for braindecode: filename=braindecode-1.2.0-py3-none-any.whl size=287261 sha256=96ea514674665fde1595ceeb9be46cad2272e241cf6cf6e40a0f359139524831
  Stored in directory: /tmp/pip-ephem-wheel-cache-ds9njgy8/wheels/b6/8b/2b/0da876924d16f36b5bdd43797902e55d90426ed71fd51642f2
Successfully built braindecode


In [8]:
import braindecode
print("Braindecode version:", braindecode.__version__)

from braindecode.models import CTNet
print("CTNet is available ✅")


Braindecode version: 1.2.0
CTNet is available ✅


In [11]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim

import importlib

import numpy as np
import os
import sys
import pickle

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [12]:
from braindecode import EEGClassifier
from braindecode.models import EEGNetv4, Deep4Net, CTNet
from braindecode.datasets import MOABBDataset
from braindecode.augmentation import FrequencyShift, GaussianNoise, AugmentedDataLoader, Compose, SmoothTimeMask, Mixup
from braindecode.training import mixup_criterion

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from skorch.helper import predefined_split
from skorch.callbacks import LRScheduler, EarlyStopping

from models.eeg_mamba_fft import create_eegmamba, EEGMamba

In [40]:
torch.manual_seed(42)
np.random.seed(42)

# Loading data for training

In [17]:
def load_subject_data_cached(subject_id):
    cache_file = f'cache/subject_{subject_id}_processed.pkl'

    if os.path.exists(cache_file):
        # Load from cache - instant!
        with open(cache_file, 'rb') as f:
            return pickle.load(f)

    # Process and cache
    train_set, test_set, train_subset, val_subset = load_subject_data(subject_id)

    os.makedirs('cache', exist_ok=True)
    with open(cache_file, 'wb') as f:
        pickle.dump((train_set, test_set, train_subset, val_subset), f)

    return train_set, test_set, train_subset, val_subset

def load_subject_data(subject_id):

    import numpy as np
    from braindecode.datasets import MOABBDataset
    from braindecode.preprocessing import Preprocessor, exponential_moving_standardize, preprocess
    from braindecode.preprocessing import create_windows_from_events
    from sklearn.model_selection import train_test_split
    from skorch.helper import SliceDataset
    from torch.utils.data import Subset

    dataset = MOABBDataset(
        dataset_name="BNCI2014_001", subject_ids=[subject_id]
    )

    #----------------------------------------------------------------------
    # After loading we preprocess

    low_cut_hz = 4.0  # low cut frequency for filtering
    high_cut_hz = 38.0  # high cut frequency for filtering
    # Parameters for exponential moving standardization
    factor_new = 1e-3
    init_block_size = 1000

    preprocessors = [
        Preprocessor("pick_types", eeg=True, meg=False, stim=False),  # Keep EEG sensors
        Preprocessor(
            lambda data, factor: np.multiply(data, factor),  # Convert from V to uV
            factor=1e6,
        ),
        Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),  # Bandpass filter
        Preprocessor(
            exponential_moving_standardize,  # Exponential moving standardization
            factor_new=factor_new,
            init_block_size=init_block_size,
        ),
    ]

    # Preprocess the data
    preprocess(dataset, preprocessors, n_jobs=-1)

    #-----------------------------------------------------------------------

    trial_start_offset_seconds = -0.5
    # Extract sampling frequency, check that they are same in all datasets
    sfreq = dataset.datasets[0].raw.info["sfreq"]
    assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
    # Calculate the window start offset in samples.
    trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

    # Create windows using braindecode function for this. It needs parameters to
    # define how windows should be used.
    windows_dataset = create_windows_from_events(
        dataset,
        trial_start_offset_samples=trial_start_offset_samples,
        trial_stop_offset_samples=0,
        preload=True,
        # verbose=0
    )

    # ----------------------------------------------------------------------
    # Split into train and test
    splitted = windows_dataset.split("session")
    train_set = splitted["0train"]  # Session train
    test_set = splitted["1test"]  # Session evaluation

    # ----------------------------------------------------------------------
    # Split into train, val subsets

    X_train = SliceDataset(train_set, idx=0)
    y_train = np.array([y for y in SliceDataset(train_set, idx=1)])
    train_indices, val_indices = train_test_split(
        X_train.indices_, test_size=0.2, shuffle=False
    )
    train_subset = Subset(train_set, train_indices)
    val_subset = Subset(train_set, val_indices)

    return train_set, test_set, train_subset, val_subset


## Data loading sanity check

In [21]:
# Run once or sanity check
subject_id = 2
train_set, test_set, train_subset, val_subset = load_subject_data_cached(subject_id)

/usr/local/lib/python3.11/dist-packages/braindecode/preprocessing/preprocess.py:71: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']


# Load and inspect model

In [22]:

from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()  # check if GPU is available, if True chooses to use it
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

# Extract number of chans and time steps from dataset
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]
n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))
classes = list(range(n_classes))

model = EEGMamba(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    n_layers = 2,
    n_experts = 8

)

model.enable_moe(True)

# Display torchinfo table describing the model
print(model)

# Send model to GPU
if cuda:
    model.cuda()

/usr/local/lib/python3.11/dist-packages/braindecode/util.py:52: UserWarning: torch.backends.cudnn.benchmark was set to True which may results in lack of reproducibility. In some cases to ensure reproducibility you may need to set torch.backends.cudnn.benchmark to False.
  warn(


Layer (type (var_name):depth-idx)             Input Shape               Output Shape              Param #                   Kernel Shape
EEGMamba (EEGMamba)                           [1, 22, 1125]             [1, 4]                    516                       --
├─STAdaptive (st_adaptive): 1-1               [1, 22, 1125]             [1, 1126, 128]            128                       --
│    └─Conv1d (proj): 2-1                     [1, 22, 1125]             [1, 128, 1125]            2,816                     [1]
├─ModuleList (layers): 1-2                    --                        --                        --                        --
│    └─BiMambaBlock (0): 2-2                  [1, 1126, 128]            [1, 1126, 128]            --                        --
│    │    └─LayerNorm (norm1): 3-1            [1, 1126, 128]            [1, 1126, 128]            256                       --
│    │    └─FFTMamba (mamba): 3-2             [1, 1126, 128]            [1, 1126, 128]            4,

# Training

# Set model hyper params

In [46]:

eegnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

deepconvnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

CTNet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW}

mamba_params = {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW}
mamba_params1 = {'lr': 0.0016, 'batch_size': 128, 'weight_decay': 5e-4, 'optimizer': torch.optim.AdamW}

n_epochs = 500

In [49]:
# Load tehe training data
train_set, test_set, train_subset, val_subset= load_subject_data_cached(2)

# Build transforms list
transforms = [
    FrequencyShift(probability=0.3, sfreq=250, max_delta_freq=0.3),
    GaussianNoise(probability=0.3, std=0.0),
    # Mixup(alpha=0.2,  beta_per_sample=True),               # ← returns (x, (y1, y2, lam))
]

# Extract model params from dataset, initialise model and set hyper-parameters
n_classes = len(torch.unique(torch.tensor([sample[1] for sample in train_subset])))
classes = list(range(n_classes))
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]

model = EEGNetv4(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    # n_layers = 2,
    # n_experts = 4

)

# Hyper params
params = eegnet_params

# Toggle this to enable mamba (For EEGMamba only)
# model.enable_moe(True)

# Enable MoE head instead of standard classifier (For EEGMamba only)
# model.use_moe = True

# # --- freeze FFT backbone ---
# for p in model.st_adaptive.parameters():
#     p.requires_grad = False

# for blk in model.layers:
#     for p in blk.parameters():  # FIX: Properly iterate through parameters
#         p.requires_grad = False

# for p in model.norm.parameters():
#     p.requires_grad = False

# --- unfreeze expert heads & gate ---

# for p in model.moe_head.parameters():
#     p.requires_grad = True

'''class MixupLoss(nn.Module):
    def __init__(self, base_criterion=None):
        super().__init__()
        # IMPORTANT: reduction='none' so we can weight per-sample
        self.base = base_criterion or nn.CrossEntropyLoss(reduction='none')

    def forward(self, preds, target):
        y1, y2, lam = target  # (B,), (B,), (B,) or scalar
        # Compute per-sample CE
        loss1 = self.base(preds, y1)          # shape: [B]
        loss2 = self.base(preds, y2)          # shape: [B]
        if not torch.is_tensor(lam):
            lam = torch.tensor(lam, device=preds.device, dtype=loss1.dtype)
        lam = lam.to(loss1.dtype)
        if lam.dim() == 0:
            # scalar lambda (rare here) -> just convex-combine scalars then mean
            return (lam * loss1 + (1 - lam) * loss2).mean()
        # per-sample lambda
        return (lam * loss1 + (1 - lam) * loss2).mean()
'''
# Create new classifier with best parameters
clf = EEGClassifier(
    model,
    # iterator_train=AugmentedDataLoader,
    # iterator_train__transforms=transforms,
    # iterator_train__shuffle=True,

    # dataset = aug_train,
    criterion=torch.nn.CrossEntropyLoss,
    # criterion__base_criterion=torch.nn.CrossEntropyLoss(reduction='none'),

    train_split=predefined_split(val_subset),  # Use all training data

    optimizer=params['optimizer'],
    optimizer__lr=params['lr'],
    optimizer__weight_decay=params['weight_decay'],
    batch_size=params['batch_size'],
    callbacks=[
        "accuracy",
        ("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=n_epochs - 1)),
        # ("early_stopping", EarlyStopping(patience=20, monitor="valid_acc")),
    ],
    device=device,
    classes=classes,
    max_epochs=n_epochs,
)

# Train on full training set
clf.fit(train_subset, y=None)

# Evaluate the model after training
y_test = test_set.get_metadata().target
test_acc = clf.score(test_set, y=y_test)

print(f"Val acc with MoE: {(test_acc * 100):.2f}%")

  epoch    train_accuracy    train_loss    valid_acc    valid_accuracy    valid_loss      lr     dur
-------  ----------------  ------------  -----------  ----------------  ------------  ------  ------
      1            0.2652        1.4030       0.2241            0.2241        1.3861  0.0010  0.1104
      2            0.3348        1.3824       0.2414            0.2414        1.3863  0.0010  0.1017
      3            0.3696        1.3668       0.2414            0.2414        1.3864  0.0010  0.0903
      4            0.4130        1.3373       0.2414            0.2414        1.3865  0.0010  0.0846
      5            0.4478        1.3086       0.2414            0.2414        1.3866  0.0010  0.0834
      6            0.4609        1.3013       0.1897            0.1897        1.3869  0.0010  0.0784
      7            0.4696        1.2904       0.1897            0.1897        1.3871  0.0010  0.0786
      8            0.4826        1.2740       0.2586            0.2586        1.3873  0.001

# Optimise model using Optuna

In [25]:
import optuna
from optuna.pruners import MedianPruner


def optimize_eegnet_hyperparams(search_subjects=[2]):  # Start with fewer subjects

    def objective(trial):
        lr = trial.suggest_float('lr', 1e-4, 6e-3, log=True)
        batch_size = trial.suggest_categorical('batch_size', [64, 128])
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3)

        subject_scores = []

        for subject_id in search_subjects:
            try:
                train_set, test_set, train_subset, val_subset= load_subject_data_cached(subject_id)

                # Get dimensions from data
                n_channels = train_subset[0][0].shape[0]
                n_times = train_subset[0][0].shape[1]
                n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))

                model = EEGMamba(
                    n_chans=n_channels,
                    n_outputs=n_classes,
                    n_times=n_times,
                    n_layers = 2,
                    n_experts = 4

                )

                clf = EEGClassifier(
                    model,

                    optimizer__lr=lr,
                    optimizer__weight_decay=weight_decay,
                    batch_size=batch_size,
                    train_split=predefined_split(val_subset),
                    device=device,
                    max_epochs=170,  # Shorter for search
                    # patience=20,
                    classes=classes,
                    verbose=0
                )

                clf.fit(train_set, y = None)
                y_test = test_set.get_metadata().target
                score = clf.score(test_set, y = y_test)
                subject_scores.append(score)

                print(f"Subject {subject_id}: {score:.3f}")

            except Exception as e:
                print(f"Error with subject {subject_id}: {e}")
                return 0.0

        avg_score = np.mean(subject_scores)

        # report intermediate value for pruning
        trial.report(avg_score, step=len(subject_scores))

        if trial.should_prune():
          optuna.TrialPruned()

        # print(f"Trial average: {avg_score:.3f}")
        return avg_score

    # Conservative
    Conservative = MedianPruner(
        n_startup_trials=10,   # Build good baseline first
        n_warmup_steps=3,      # Wait for 3 subjects
        interval_steps=1       # Check after each subject
    )

    # Aggressive (limited time)
    Aggressive = MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=2,
        interval_steps=1
    )

    study = optuna.create_study(direction='maximize', pruner=Conservative)
    study.optimize(objective, n_trials=40)  # Start with fewer trials

    return study.best_params, study.best_value

# Use it
best_params, best_score = optimize_eegnet_hyperparams()
print(f"Best EEGNet params: {best_params}")
print(f"Best average accuracy: {best_score:.3f}")

[I 2025-08-18 09:10:14,130] A new study created in memory with name: no-name-bc4e76f3-3b77-4e23-9025-7fdfaccf0976
[I 2025-08-18 09:10:58,077] Trial 0 finished with value: 0.28125 and parameters: {'lr': 0.0020705610595518407, 'batch_size': 128, 'weight_decay': 0.0005271279923227597}. Best is trial 0 with value: 0.28125.


Subject 2: 0.281


[I 2025-08-18 09:11:42,402] Trial 1 finished with value: 0.2847222222222222 and parameters: {'lr': 0.00031654770337600737, 'batch_size': 128, 'weight_decay': 0.0006342530181878795}. Best is trial 1 with value: 0.2847222222222222.


Subject 2: 0.285


[I 2025-08-18 09:12:26,754] Trial 2 finished with value: 0.24305555555555555 and parameters: {'lr': 0.00037826168946949106, 'batch_size': 64, 'weight_decay': 0.0008252711449553231}. Best is trial 1 with value: 0.2847222222222222.


...

Subject 2: 0.271


[I 2025-08-18 09:39:27,429] Trial 38 finished with value: 0.22569444444444445 and parameters: {'lr': 0.0004231809426550327, 'batch_size': 128, 'weight_decay': 0.0005168077405286721}. Best is trial 10 with value: 0.3020833333333333.


Subject 2: 0.226


[I 2025-08-18 09:40:12,531] Trial 39 finished with value: 0.25 and parameters: {'lr': 0.00024435444571985215, 'batch_size': 64, 'weight_decay': 0.0004623630452806089}. Best is trial 10 with value: 0.3020833333333333.


Subject 2: 0.250
Best EEGNet params: {'lr': 0.00017482485340478428, 'batch_size': 64, 'weight_decay': 0.000996737023095664}
Best average accuracy: 0.302


In [26]:
nbp = best_params
nbp

{'lr': 0.00017482485340478428,
 'batch_size': 64,
 'weight_decay': 0.000996737023095664}

In [ ]:
param_grid_eegnet = {
    "optimizer__lr": [0.001, 0.000625],
    "optimizer__weight_decay": [0.005, 0.00001],
    "batch_size": [64, 128],
}
param_grid_moe = {
    "optimizer__lr": [0.0016, 0.0032, 0.0064],
    "optimizer__weight_decay": [5e-5, 1e-4, 5e-4],
    "batch_size": [64, 128],
}
param_grid = param_grid_moe

# By setting n_jobs=-1, grid search is performed
# with all the processors, in this case the output of the training
# process is not printed sequentially
search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=train_val_split,
    return_train_score=True,
    scoring="accuracy",
    refit=True,
    verbose=1,
    error_score="raise",
    n_jobs=1,
)

search.fit(X_train, y_train)
search_results = pd.DataFrame(search.cv_results_)

best_run = search_results[search_results["rank_test_score"] == 1].squeeze()

best_parameters = best_run["params"]

In [22]:
subject_id = "__all_"

torch.save(eegmamba_clf, f'models/eegmamba_subject{subject_id}_baseline.pt')

def save_baseline_model(clf, model_name, subject_id, save_dir='models/baselines'):
    """Save trained baseline for adversarial testing."""
    os.makedirs(save_dir, exist_ok=True)

    mean, std = compute_zscore_stats(train_set)
    norm_info = dump_norm_modules(model)

    # Create comprehensive checkpoint
    checkpoint = {
        'clf': clf,  # Full classifier
        'model_state_dict': clf.module_.state_dict(),
        'model_type': model_name,
        'subject_id': subject_id,
        'model_params': {
            "version": "eegmamba_baseline_v1",
            'n_chans': clf.module_.n_chans,
            'n_outputs': clf.module_.n_outputs,
            'n_times': clf.module_.n_times,
            "sfreq": float(sfreq),
            "zscore": {
                "per_channel_mean": mean,
                "per_channel_std": std,
                "note": "computed over training windows",
            },
        "normalization_modules": norm_info,
        },
        'training_history': clf.history,
        'final_train_acc': clf.history[-1]['train_accuracy'],
        'final_valid_acc': clf.history[-1]['valid_accuracy'] if 'valid_accuracy' in clf.history[-1] else None,
    }

    filename = f"{model_name}_subject{subject_id}_baseline.pt"
    filepath = os.path.join(save_dir, filename)
    torch.save(checkpoint, filepath)

    print(f"Saved {model_name} baseline for subject {subject_id} to {filepath}")
    return filepath

# Usage
# save_baseline_model(eegnet_clf, 'EEGNet', subject_id=1)
# save_baseline_model(ctnet_clf, 'CTNet', subject_id=1)
save_baseline_model(eegmamba_clf, 'EEGMamba', subject_id)

Saved EEGMamba baseline for subject 3 to models/baselines/EEGMamba_subject03_baseline.pt


'models/baselines/EEGMamba_subject03_baseline.pt'

In [116]:


# Extract best parameters from GridSearch
# best_params = search.best_params_

# Create new classifier with best parameters
final_clf = EEGClassifier(
    model,
    iterator_train=AugmentedDataLoader,
    iterator_train__transforms=transforms,
    criterion=torch.nn.CrossEntropyLoss,
    optimizer=torch.optim.AdamW,
    train_split=predefined_split(tr),  # Use all training data
    optimizer__lr=best_params['optimizer__lr'],  # Use best lr other parameters from best_params
    optimizer__weight_decay=best_params['optimizer__weight_decay'],
    batch_size=best_params['batch_size'],  # Use best batch size
    callbacks=[
        "accuracy",
        ("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=n_epochs - 1)),
    ],
    device=device,
    classes=classes,
    max_epochs=n_epochs,
)

# Train on full training set
final_clf.fit(train_set, y=None)

# evaluated the model after training
y_test = test_set.get_metadata().target
test_acc = final_clf.score(test_set, y=y_test)
print(f"Test acc: {(test_acc * 100):.2f}%")

  epoch    train_accuracy    train_loss    valid_acc    valid_accuracy    valid_loss      lr     dur
-------  ----------------  ------------  -----------  ----------------  ------------  ------  ------
      1            0.2500        1.3973       0.2500            0.2500        1.3955  0.0016  0.7191
      2            0.2986        1.4050       0.3264            0.3264        1.3882  0.0016  0.6557
      3            0.4236        1.3885       0.4132            0.4132        1.3772  0.0016  0.7533
      4            0.2500        1.3822       0.2500            0.2500        1.3893  0.0016  0.7495
      5            0.3229        1.3921       0.3472            0.3472        1.3633  0.0016  0.7052
      6            0.3160        1.3647       0.2465            0.2465        1.3661  0.0016  0.6481
      7            0.3542        1.3632       0.3507            0.3507        1.3436  0.0016  0.6753
      8            0.3854        1.3478       0.3819            0.3819        1.3290  0.001

In [57]:
print(train_set.get_metadata().subject.values)


[1 1 1 ... 9 9 9]
